<a href="https://colab.research.google.com/github/seethaladevi2024-cpu/OIBSIP/blob/main/UnitConverter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Flask Unit Converter Web Application for Google Colab
# This app provides a web interface for unit conversions
# Uses Google Colab's native method to expose the web app

# Step 1: Install required packages
print("📦 Installing required packages...")
!pip install flask flask-cors -q

# Step 2: Import necessary libraries
from flask import Flask, render_template_string, request, jsonify
from flask_cors import CORS
import threading
from google.colab.output import eval_js

# Step 3: Define the HTML template with embedded CSS
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Unit Converter</title>
    <style>
        /* Reset and base styles */
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            padding: 20px;
        }

        /* Main container */
        .container {
            background: white;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0, 0, 0, 0.3);
            padding: 40px;
            max-width: 500px;
            width: 100%;
        }

        /* Header */
        h1 {
            color: #333;
            text-align: center;
            margin-bottom: 10px;
            font-size: 28px;
        }

        .subtitle {
            text-align: center;
            color: #666;
            margin-bottom: 30px;
            font-size: 14px;
        }

        /* Form elements */
        .form-group {
            margin-bottom: 25px;
        }

        label {
            display: block;
            color: #555;
            font-weight: 600;
            margin-bottom: 8px;
            font-size: 14px;
        }

        input[type="number"],
        select {
            width: 100%;
            padding: 12px 15px;
            border: 2px solid #e0e0e0;
            border-radius: 10px;
            font-size: 16px;
            transition: all 0.3s ease;
            background: #f9f9f9;
        }

        input[type="number"]:focus,
        select:focus {
            outline: none;
            border-color: #667eea;
            background: white;
        }

        /* Button */
        .btn-convert {
            width: 100%;
            padding: 15px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 10px;
            font-size: 18px;
            font-weight: 600;
            cursor: pointer;
            transition: transform 0.2s ease, box-shadow 0.2s ease;
            margin-top: 10px;
        }

        .btn-convert:hover {
            transform: translateY(-2px);
            box-shadow: 0 10px 25px rgba(102, 126, 234, 0.4);
        }

        .btn-convert:active {
            transform: translateY(0);
        }

        /* Result area */
        .result-area {
            margin-top: 30px;
            padding: 20px;
            background: #f0f4ff;
            border-radius: 10px;
            min-height: 80px;
            display: flex;
            align-items: center;
            justify-content: center;
        }

        .result-text {
            color: #333;
            font-size: 18px;
            font-weight: 600;
            text-align: center;
        }

        .error {
            color: #e74c3c;
        }

        .success {
            color: #27ae60;
        }

        /* Icon */
        .icon {
            font-size: 40px;
            text-align: center;
            margin-bottom: 20px;
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="icon">🔄</div>
        <h1>Unit Converter</h1>
        <p class="subtitle">Convert between different units easily</p>

        <form id="converterForm">
            <div class="form-group">
                <label for="value">Enter Value:</label>
                <input
                    type="number"
                    id="value"
                    name="value"
                    step="any"
                    placeholder="e.g., 100"
                    required
                >
            </div>

            <div class="form-group">
                <label for="conversion">Select Conversion Type:</label>
                <select id="conversion" name="conversion" required>
                    <option value="cm_to_m">Centimeters to Meters</option>
                    <option value="m_to_cm">Meters to Centimeters</option>
                    <option value="g_to_kg">Grams to Kilograms</option>
                    <option value="kg_to_g">Kilograms to Grams</option>
                </select>
            </div>

            <button type="submit" class="btn-convert">Convert</button>
        </form>

        <div class="result-area" id="resultArea">
            <p class="result-text" id="resultText">Enter a value and click Convert</p>
        </div>
    </div>

    <script>
        // Handle form submission
        document.getElementById('converterForm').addEventListener('submit', async function(e) {
            e.preventDefault();

            const value = document.getElementById('value').value;
            const conversion = document.getElementById('conversion').value;
            const resultText = document.getElementById('resultText');

            // Show loading state
            resultText.textContent = '⏳ Converting...';
            resultText.className = 'result-text';

            try {
                // Send POST request to backend
                const response = await fetch('/convert', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({
                        value: parseFloat(value),
                        conversion: conversion
                    })
                });

                const data = await response.json();

                // Display result or error
                if (data.error) {
                    resultText.textContent = '❌ ' + data.error;
                    resultText.className = 'result-text error';
                } else {
                    resultText.textContent = '✅ ' + data.result;
                    resultText.className = 'result-text success';
                }
            } catch (error) {
                resultText.textContent = '❌ An error occurred. Please try again.';
                resultText.className = 'result-text error';
            }
        });
    </script>
</body>
</html>
"""

# Step 4: Initialize Flask application
app = Flask(__name__)
CORS(app)  # Enable CORS for Colab

# Step 5: Define conversion functions
def convert_units(value, conversion_type):
    """
    Perform unit conversion based on the selected type.

    Args:
        value: Numeric value to convert
        conversion_type: Type of conversion to perform

    Returns:
        Tuple of (result, formatted_string)
    """
    conversions = {
        'cm_to_m': (value / 100, f"{value} cm = {value / 100:.4f} m"),
        'm_to_cm': (value * 100, f"{value} m = {value * 100:.2f} cm"),
        'g_to_kg': (value / 1000, f"{value} g = {value / 1000:.4f} kg"),
        'kg_to_g': (value * 1000, f"{value} kg = {value * 1000:.2f} g")
    }

    return conversions.get(conversion_type)

# Step 6: Define routes

@app.route('/')
def home():
    """Render the main page with the HTML template"""
    return render_template_string(HTML_TEMPLATE)

@app.route('/convert', methods=['POST'])
def convert():
    """
    Handle conversion requests from the frontend.
    Validates input and returns the converted result.
    """
    try:
        # Get JSON data from request
        data = request.get_json()
        value = data.get('value')
        conversion_type = data.get('conversion')

        # Validate input value
        if value is None:
            return jsonify({'error': 'Please enter a value'}), 400

        if value < 0:
            return jsonify({'error': 'Please enter a positive number'}), 400

        # Perform conversion
        result = convert_units(value, conversion_type)

        if result is None:
            return jsonify({'error': 'Invalid conversion type'}), 400

        # Return formatted result
        return jsonify({'result': result[1]})

    except (ValueError, TypeError) as e:
        return jsonify({'error': 'Invalid input. Please enter a valid number.'}), 400
    except Exception as e:
        return jsonify({'error': 'An error occurred during conversion.'}), 500

# Step 7: Run the app using Google Colab's output feature
def run_with_colab():
    """Run Flask app and get the public URL from Colab"""
    import time

    # Start Flask in a background thread
    print("🚀 Starting Flask server...")
    threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False), daemon=True).start()

    # Wait for Flask to start
    time.sleep(3)

    # Get the public URL from Colab
    print("🌐 Getting public URL from Google Colab...")
    try:
        public_url = eval_js("google.colab.kernel.proxyPort(5000)")
        print(f"\n✅ SUCCESS! Your app is running!")
        print(f"🔗 Public URL: {public_url}")
        print(f"\n📱 Click the link above to access your Unit Converter web app!")
        print(f"⚠️  Note: Keep this cell running to maintain the connection.")
        print(f"\n💡 The URL will remain active as long as the Colab session is running.")
    except Exception as e:
        print(f"\n⚠️  Could not automatically get public URL.")
        print(f"📋 Manual steps:")
        print(f"1. The Flask server is running on port 5000")
        print(f"2. Click on the 'Open in new tab' icon that appears next to the cell")
        print(f"3. Or look for the link in the cell output area")

# Execute the function
run_with_colab()

📦 Installing required packages...
🚀 Starting Flask server...
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


🌐 Getting public URL from Google Colab...

✅ SUCCESS! Your app is running!
🔗 Public URL: https://5000-m-s-obglf00qzt42-b.us-west3-1.prod.colab.dev

📱 Click the link above to access your Unit Converter web app!
⚠️  Note: Keep this cell running to maintain the connection.

💡 The URL will remain active as long as the Colab session is running.
